# L4 36 — validate the context-aware Qwen 3B link

Runs two complementary frozen-link checks:

1. neutral held-out fidelity, to detect regression from the original capability; and
2. action-distribution fidelity on 384 arena deployment snapshots that are disjoint from all 256 snapshots used to diagnose the old link.

No weights are updated. Both stages checkpoint results to Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = 'cad02ccf62079bdc94acf0d607a1d17200093106'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
SOURCE_JOB_ID = 'faithful-qwen3b-t4-001'
JOB_ID = 'contextual-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
ROOT = pathlib.Path('/content/drive/MyDrive/rival-arena-l4')
SOURCE_DIR = ROOT/SOURCE_JOB_ID
JOB_DIR = ROOT/JOB_ID
MATCHES = SOURCE_DIR/'arena_ipd_confirmatory_v3/matches.jsonl'
EXCLUSIONS = SOURCE_DIR/'arena_context_fidelity_v1/snapshots.jsonl'
for required in (JOB_DIR/'faithful_link.pt', MATCHES, EXCLUSIONS):
    assert required.exists(), f'Missing required Drive artifact: {required}'
print('Validating:', JOB_DIR)

In [ ]:
neutral = [
    'python', 'scripts/validate_link.py',
    '--model', MODEL,
    '--job-dir', JOB_DIR,
    '--job-id', JOB_ID,
    '--examples', '256',
    '--probe-examples', '240',
    '--generation-samples', '12',
]
print(' '.join(map(str, neutral)))
subprocess.run(list(map(str, neutral)), cwd=WORK/'followup-representational', check=True, env=os.environ)

In [ ]:
deployment = [
    'python', 'scripts/diagnose_arena_fidelity.py',
    '--model', MODEL,
    '--job-dir', JOB_DIR,
    '--job-id', JOB_ID,
    '--matches', MATCHES,
    '--exclude-snapshots', EXCLUSIONS,
    '--samples', '384',
    '--seed', '20260726',
]
print(' '.join(map(str, deployment)))
subprocess.run(list(map(str, deployment)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected outputs are `validation_report.json` and `arena_context_fidelity_v1/` inside `MyDrive/rival-arena-l4/contextual-qwen3b-t4-001/`. Do not launch the pilot until the deployment report is reviewed.